In [6]:
import re
import os
import pandas as pd
import numpy as np

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))

In [7]:
def split_event(event: str):
    """
    Parse an event string like '10,000 metres, Men (Olympic)'
    or 'Singles, Handicap, Men (Olympic (non-medal))'
    Returns (event_name, gender, handicap).
    """
    if pd.isna(event):
        return pd.NA, pd.NA, False

    # Strip ALL parenthetical suffixes, including nested ones like
    # '(Olympic (non-medal))' — loop until nothing left to remove
    cleaned = event
    while True:
        new = re.sub(r'\s*\([^()]*\)', '', cleaned).strip()
        if new == cleaned:
            break
        cleaned = new

    # Detect and remove 'Handicap' token (case-insensitive)
    handicap = bool(re.search(r',\s*Handicap\b', cleaned, re.IGNORECASE))
    cleaned  = re.sub(r',\s*Handicap\b', '', cleaned, flags=re.IGNORECASE).strip()

    # The gender token is always the LAST comma-separated chunk
    # e.g. "Singles, Men" / "10,000 metres, Women" / "4 × 100 metres relay, Mixed"
    # but NOT "10,000 metres" where there's no gender suffix
    gender_pattern = re.compile(r',\s*(Men|Women|Mixed)\s*$', re.IGNORECASE)
    m = gender_pattern.search(cleaned)

    if m:
        gender     = m.group(1).capitalize()
        event_name = cleaned[:m.start()].strip()
    else:
        gender     = pd.NA
        event_name = cleaned

    return event_name, gender, handicap


In [8]:
input_path = os.path.join(PROJECT_ROOT, 'data', '1-raw_data', 'all_participations.csv')
output_path = os.path.join(PROJECT_ROOT, 'data', '2-cleaned_data', 'all_participations.csv')
df = pd.read_csv(input_path)

if "event" not in df.columns:
    print(f"Error: no 'event' column found. Columns: {list(df.columns)}")

df[["event_name", "gender", "handicap"]] = df["event"].apply(
    lambda e: pd.Series(split_event(e))
)

# Drop original event column and reorder: put new cols right after 'discipline'
df = df.drop(columns=["event"])
cols = list(df.columns)
if "discipline" in cols:
    insert_at = cols.index("discipline") + 1
    for col in reversed(["event_name", "gender"]):
        cols.insert(insert_at, cols.pop(cols.index(col)))
df = df[cols]

df.to_csv(output_path, index=False)
print(f"✓ {len(df):,} rows written to '{output_path}'")
print(f"  Unique genders found:      {sorted(df['gender'].dropna().unique())}")
print(f"  Rows with no gender token: {df['gender'].isna().sum()}")
print(df[["year","discipline", "event_name", "gender"]].head(8).to_string(index=False))


✓ 308,408 rows written to 'cleaned_data/all_participations.csv'
  Unique genders found:      ['Men', 'Mixed', 'Women']
  Rows with no gender token: 19998
  year discipline event_name gender
1912.0     Tennis    Singles    Men
1912.0     Tennis    Doubles    Men
1920.0     Tennis    Singles    Men
1920.0     Tennis    Doubles  Mixed
1920.0     Tennis    Doubles    Men
1996.0     Tennis    Singles    Men
1996.0     Tennis    Doubles    Men
1924.0     Tennis    Singles    Men


In [9]:
def normalize_time(val) -> str:
    """
    Convert any swim-time format to HH:MM:SS.ffffff
    Handles:
      - '22:36.4est'  → '00:22:36.400000'
      - '51.98'       → '00:00:51.980000'
      - '1:52:36.4'   → '01:52:36.400000'
    """
    if pd.isna(val) or val == "Did not finish" or val =="Did not start" or val == "Disqualified":
        return pd.NA

    # Strip trailing non-numeric suffixes (est, h, etc.)
    s = re.sub(r'[a-zA-Z]+$', '', str(val).strip())

    if not s:  # was either empty or pure letters like 'est'
        return pd.NA

    parts = s.split(':')

    if len(parts) == 1:
        h, m, sec = 0, 0, float(parts[0])
    elif len(parts) == 2:
        h, m, sec = 0, int(parts[0]), float(parts[1])
    elif len(parts) == 3:
        h, m, sec = int(parts[0]), int(parts[1]), float(parts[2])
    else:
        return pd.NA

    td = pd.Timedelta(hours=h, minutes=m, seconds=sec)
    total_seconds = td.total_seconds()

    return total_seconds

In [18]:
input_path = os.path.join(PROJECT_ROOT, 'data', '1-raw_data', 'Olympic_Swimming_Results_1912to2020.csv')
output_path = os.path.join(PROJECT_ROOT, 'data', '2-cleaned_data', 'swimming.csv')
df_swim = pd.read_csv(input_path)


df_swim["Rank"]    = pd.to_numeric(df_swim["Rank"], errors="coerce")
df_swim["Results"] = df_swim["Results"].apply(normalize_time)
df_swim["Results"] = pd.to_numeric(df_swim["Results"], errors="coerce")

#  Rename the column
df_swim = df_swim.rename(columns={"Distance (in meters)": "Distance"})

# Remove the trailing 'm' and convert to numeric
df_swim["Distance"] = (
    df_swim["Distance"]
    .str.replace("m", "", regex=False)
    .str.strip()
)

df_swim["Distance"] = pd.to_numeric(df_swim["Distance"], errors="coerce")

event_cols = ["Athlete", "Distance", "Stroke"]

print(df_swim.columns)
df_swim = df_swim.sort_values(["Athlete", "Distance", "Stroke", "Year"])

df_swim["Experience"] = df_swim.groupby(event_cols).cumcount()

df_swim["Previous_Best"] = (
    df_swim.groupby(event_cols)["Results"]
      .transform(lambda x: x.shift().expanding().min())
)

df_swim["Previous_Average"] = (
    df_swim.groupby(event_cols)["Results"]
      .transform(lambda x: x.shift().expanding().mean())
)

df_swim[df_swim["Relay?"] == 0].to_csv(output_path, index=False)

Index(['Location', 'Year', 'Distance', 'Stroke', 'Relay?', 'Gender', 'Team',
       'Athlete', 'Results', 'Rank'],
      dtype='object')
